# Get start with the Colette Python API

## Install the ipywidgets package if not already installed

In [ ]:
# !pip install -U ipywidgets

## Import necessary libraries

In [ ]:
import json
import base64
from io import BytesIO
from PIL import Image
from IPython.display import display
import re

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from colette.jsonapi import JSONApi
from colette.apidata import APIData

## Get paths and initialize variables

In [ ]:
# Get the root path of the colette package
import os
colette_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
# print(f'Colette root path: {colette_root}')

In [ ]:
documents_dir = os.path.join(colette_root, 'docs/pdf') # where the input documents are located
app_dir = os.path.join(colette_root, 'app_colette') # where to store the app
models_dir = os.path.join(colette_root, 'models') # where the models are located

config_file = os.path.join(colette_root, 'src/colette/config/vrag_default.json')
index_file = os.path.join(colette_root, 'src/colette/config/vrag_default_index.json')

app_name = 'app_colette' # name of the app

In [ ]:
colette_api = JSONApi()

## Get config files and set parameters

In [ ]:
# read the configuration file
with open(config_file, 'r') as f:
    create_config = json.load(f)
with open(index_file, 'r') as f:
    index_config = json.load(f)

create_config['app']['repository'] = app_dir
create_config['app']['models_repository'] = models_dir
index_config['parameters']['input']['data'] = [documents_dir]
# Index in hybrid mode so both embedding retrieval and text search retrieval data are available.
# All retrieval_mode values ('embedding_retrieval', 'text_search_retrieval', 'hybrid') then work without re-indexing.
index_config['parameters']['input']['rag']['retrieval_mode'] = 'hybrid'
#index_config['parameters']['input']['rag']['reindex'] = False # if True, the RAG will be reindexed


## Create the Service API client

In [ ]:
api_data_create = APIData(**create_config)
colette_api.service_create(app_name, api_data_create)

## Index the documents

In [ ]:
# index the documents
api_data_index = APIData(**index_config)
colette_api.service_index(app_name, api_data_index)

## Query the documents

Note the optional 'crop_label' parameter to filter the sources by crop label.

The default crop labels are: 'text', 'table', 'figure'

In [ ]:
# query the vision RAG
query_api_msg = {
    'parameters': {
        'input': {
            'message': 'What are the identified sources of errors ?',
            # 'crop_label': 'text',  # optional, to specify a crop label
        }
    }
}

query_data = APIData(**query_api_msg)
response = colette_api.service_predict(app_name, query_data)

In [ ]:
print(response.output)

In [ ]:
# Function to extract and display image from base64 data URI
def display_image_from_data_uri(data_uri):
    # Extract base64 string (remove 'data:image/png;base64,' prefix)
    base64_str = re.sub('^data:image/.+;base64,', '', data_uri)
    
    # Decode base64 string
    image_data = base64.b64decode(base64_str)
    
    # Create PIL Image
    image = Image.open(BytesIO(image_data))
    
    # Display image
    display(image)
    
    return image

In [ ]:
# Display all images in your context
for item in response.sources['context']:
    print(f"Key: {item['key']}")
    print(f"Distance: {item['distance']}")
    print(f'crop_label: {item.get("crop_label", "N/A")}')
    image = display_image_from_data_uri(item['content'])
    print(f"Image size: {image.size}")
    print("-" * 50)

## Retrieval Modes: Embedding, Text Search, and Hybrid

Colette supports three retrieval modes in `parameters.input.rag.retrieval_mode`:
- `embedding_retrieval`: visual embedding retrieval only (default) -> results in `sources['context']`
- `text_search_retrieval`: lexical text search (BM25) only -> results in `sources['text_context']`
- `hybrid`: combined retrieval -> results in both `sources['context']` and `sources['text_context']`

You can set this at index-time and override per request.


In [ ]:
# The index above was already created with retrieval_mode='hybrid' (embedding + text search).
# There is no need to re-index - all three retrieval modes work against the same index.
#
# If you ever need to change the indexing mode later, you would call service_index again:
#   index_config_new_mode = json.loads(json.dumps(index_config))
#   index_config_new_mode['parameters']['input']['rag']['retrieval_mode'] = 'embedding_retrieval'
#   colette_api.service_index(app_name, APIData(**index_config_new_mode))
# Note: re-indexing recreates the ChromaDB collection, so the service must also be recreated
# afterwards to pick up the new collection reference.


In [ ]:
# Example: per-request override to text-search only
# Only retrieval_mode is overridden; text_search_engine_* values come from JSON defaults
query_text_search = {
    'parameters': {
        'input': {
            'message': 'What are the identified sources of errors ?',
            'rag': {
                'retrieval_mode': 'text_search_retrieval',
            }
        }
    }
}
response_text_search = colette_api.service_predict(app_name, APIData(**query_text_search))

In [ ]:
print(response_text_search.output)

In [ ]:
# With retrieval_mode='text_search_retrieval', embedding retrieval is skipped - sources['context'] is empty.
# Text search results are in sources['text_context'] instead.
for hit in response_text_search.sources.get('text_context', []):
    print(f"source: {hit['source']}")
    print(f"page: {hit.get('page_number')}")
    print(f"score: {hit.get('score', 'n/a'):.3f}")
    print(hit['content'][:200].replace('\n', ' '))
    print("-" * 50)


In [ ]:
# If retrieval_mode includes text search, inspect text_context in sources
response_hybrid = colette_api.service_predict(
    app_name,
    APIData(**{'parameters': {'input': {'message': 'What are the identified sources of errors ?', 'rag': {'retrieval_mode': 'hybrid'}}}})
)
for hit in response_hybrid.sources.get('text_context', []):
    print(f"{hit['source']} page {hit.get('page_number')} score={hit.get('score', 'n/a')}")
    print(hit['content'][:200])
    print()


In [ ]:
print(response_hybrid.output)

In [ ]:
for hit in response_hybrid.sources.get('text_context', []):
    print(f"source: {hit['source']}")
    print(f"page: {hit.get('page_number')}")
    print(f"score: {hit.get('score', 'n/a'):.3f}")
    print(hit['content'][:200].replace('\n', ' '))
    print("-" * 50)

In [ ]:
# Display all images in your context
for item in response_hybrid.sources['context']:
    print(f"Key: {item['key']}")
    print(f"Distance: {item['distance']}")
    print(f'crop_label: {item.get("crop_label", "N/A")}')
    image = display_image_from_data_uri(item['content'])
    print(f"Image size: {image.size}")
    print("-" * 50)